# GRU

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import matplotlib.pyplot as plt

In [2]:
from gen_seq_data import SequenceDataset
from vech import vech, unvech

## Modellieren von ${r_t}^2$

In [ ]:
tickers = "^GSPC BTC-USD GC=F"  # S&P 500 index, Bitcoin, Gold
data = yf.download(tickers, period="max")
close = data["Close"].copy()

In [ ]:
logret = np.log(close).diff() # log(P_t) - log(P_{t-1})) = log(P_t / P_{t-1})

# Aufräumen: erste Zeile NaN durch diff, ggf. weitere NaNs durch Datenlücken
logret = logret.dropna()

### Train, Val, Test Split

In [ ]:
n = len(logret)
print(n)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)
print(train_end, val_end)

train_df = logret.iloc[:train_end]
val_df   = logret.iloc[train_end:val_end]
test_df  = logret.iloc[val_end:]

## Scaling the data

- vielleicht ```*100``` für bessere Performance?

In [ ]:
feature_cols = tickers.split(" ")

mu = train_df[feature_cols].mean()
sigma = train_df[feature_cols].std()

def normalize(df, mu, sigma):
    return (df[feature_cols] - mu) / sigma

train = normalize(train_df, mu, sigma)
val = normalize(val_df, mu, sigma)
test = normalize(test_df, mu, sigma)# würde man Test mit Test Mean und Std normalizen, wäre das Data Leakage, da Informationen aus der Zukunft genutzt würden, die eig. nicht vorliegen sollten

## Generate Sequence Dataset & Dataloader

In [ ]:
lookback = 60
batchsize = 128

In [ ]:
train = SequenceDataset(train[feature_cols].values, train[feature_cols].values, lookback)
val = SequenceDataset(val[feature_cols].values, val[feature_cols].values, lookback)
test = SequenceDataset(test[feature_cols].values, test[feature_cols].values, lookback)

In [ ]:
train_loader = DataLoader(train, batch_size=batchsize, shuffle=False)
val_loader   = DataLoader(val, batch_size=batchsize, shuffle=False)
test_loader  = DataLoader(test, batch_size=batchsize, shuffle=False)